### Volume bars as iid normal: fixed vs dynamically adjusted thresholds

Reads trade ticks from the Parquet catalog one day at a time and builds volume bars for every
candidate specification in a single pass. Only bars are held in memory, never ticks.

Two families of specification:

- **`fix_T`** — constant threshold of `T` ETH per bar.
- **`dyn_N{n}_W{w}`** — threshold recalibrated daily as `ADVV(d-w .. d-1) / n`, targeting `n`
  bars/day. Strictly trailing, so no look-ahead enters the bar boundaries.

---

#### A warning about ranking on kurtosis alone

Excess kurtosis falls monotonically as the threshold grows — each bar aggregates more trades, so
the central limit theorem pulls the return distribution toward normal. **Ranking by kurtosis will
therefore always select the largest threshold in the grid**, which is not a finding, it is an
artefact of the criterion. Pushing the grid higher will keep "improving" normality forever, at
the cost of ever-slower bars.

The useful question is not *which threshold is most normal* but *what is the fastest bar that is
normal enough*. Cell 9 reports the **smallest threshold meeting a kurtosis tolerance** rather than
the argmin, and the retest analysis at the end uses the same rule. Set `KURT_TOL` to what your
models can actually tolerate.

#### Period length and estimator noise

Sample excess kurtosis has standard error of about `sqrt(24/n)`. At 20 bars/day a month gives
~600 bars, so monthly kurtosis carries roughly ±0.2 of pure noise — comparable to the tolerance
itself. `PERIOD='M'` is used because the retest walk-forward needs that resolution, but for
judging the large thresholds alone, `PERIOD='Q'` gives a far steadier read. Genuine spikes to 5+
are real fat-tailed months, not noise; the `bars` column tells you which is which.

---

#### Day-seam correction

Building bars day by day equals one global cumsum only if boundaries falling *between* the last
tick of one day and the first of the next are recovered explicitly — `np.diff` inside a day
cannot see them. Roughly one in three day boundaries produces such a close, so at ~30 bars/day
this reaches ~1% of bars. `close_bars` handles it.

The accumulator is carried as a **fraction of a bar** rather than raw ETH, so a threshold that
changes between days stays consistent. With a fixed threshold the two are identical.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import stats

TICKS = Path.cwd().parent / 'nautilusDataCatalog' / 'data' / 'trade_tick' / 'ETHUSDT-LINEAR.BYBIT'

FIXED_THRESHOLDS = list(range(20_000, 50_001, 2_500))  # ETH per bar
N_TARGETS = [20, 30, 40, 50]                           # dynamic: target bars/day
W_WINDOWS = [21, 63]                                   # dynamic: ADVV trailing days

PERIOD = 'M'          # monthly -- needed for the retest walk-forward in the last cell
LAST_N_DAYS = None    # e.g. 400 to test quickly; both passes honour it
KURT_TOL = 0.5        # "normal enough" tolerance for threshold selection
LOOKBACK = 6          # periods of history used to select a threshold
MIN_BARS = 200        # periods with fewer bars than this are dropped for that spec
PLOT_MAX = 200_000
PLOT_PERIODS = 3      # how many of the most recent periods to draw distributions for

SCALE_128 = 1e16      # Nautilus high-precision fixed-point
SCALE_64 = 1e9        # standard-precision fixed-point

# (name, kind, param_a, param_b)
SPECS = ([(f'fix_{t}', 'fixed', t, None) for t in FIXED_THRESHOLDS]
         + [(f'dyn_N{n}_W{w}', 'dyn', n, w) for n in N_TARGETS for w in W_WINDOWS])
print(f'{len(SPECS)} specs: {len(FIXED_THRESHOLDS)} fixed, {len(SPECS) - len(FIXED_THRESHOLDS)} dynamic')

In [ ]:
def period_of(name):
    y, m = int(name[:4]), int(name[5:7])
    if PERIOD == 'M':
        return f'{y}-{m:02d}'
    return f'{y}Q{(m - 1) // 3 + 1}' if PERIOD == 'Q' else str(y)


files = sorted(TICKS.glob('*.parquet'))
if LAST_N_DAYS:
    files = files[-LAST_N_DAYS:]

groups = {}
for f in files:
    groups.setdefault(period_of(f.name), []).append(f)

print(f'{len(files)} days -> {len(groups)} periods: {list(groups)[0]} .. {list(groups)[-1]}')

In [ ]:
def decode(col):
    """Fixed-point price/size column -> float64.

    128-bit columns arrive as 16-byte binary, read here as two uint64 halves.
    Recombining in float64 loses far less than any real tick increment.
    """
    if pa.types.is_fixed_size_binary(col.type):
        a = col.combine_chunks()
        a = a.chunk(0) if isinstance(a, pa.ChunkedArray) else a
        raw = np.frombuffer(a.buffers()[1], dtype=np.uint64, count=2 * len(a))
        return (raw[0::2] + raw[1::2] * 2.0 ** 64) / SCALE_128
    v = col.to_numpy(zero_copy_only=False)
    return v / SCALE_64 if np.issubdtype(v.dtype, np.integer) else v.astype(np.float64)


def read_day(path):
    """ts_event (ns), price, size as float arrays."""
    t = pq.read_table(path, columns=['ts_event', 'price', 'size'])
    ts = t['ts_event'].to_numpy(zero_copy_only=False)
    return ts, decode(t['price']), decode(t['size'])


def day_of(path):
    return pd.Timestamp(path.name[:10])


def close_bars(ts, px, sz, threshold, carry, prev=None):
    """Volume bars. Returns (bar ts, bar close, new carry).

    `carry` is a *fraction* of a bar in [0, 1), so a threshold that changes between
    days stays consistent. `prev` is the previous day's final (ts, price): a bar can
    close exactly on the day seam, which np.diff cannot see from inside either day,
    and a single global cumsum attributes that close to the previous day's last tick.

    A trade straddling a boundary is assigned whole rather than split, which is
    immaterial at these tick counts.
    """
    cum = carry + np.cumsum(sz / threshold)
    floors = np.floor(cum)

    out_ts, out_px = [], []
    if floors[0] > 0 and prev is not None:
        out_ts.append(np.array([prev[0]], dtype=ts.dtype))
        out_px.append(np.array([prev[1]], dtype=px.dtype))

    end = np.flatnonzero(np.diff(floors))  # last tick of each completed bar
    out_ts.append(ts[end])
    out_px.append(px[end])

    return np.concatenate(out_ts), np.concatenate(out_px), float(cum[-1] % 1.0)

In [ ]:
# Sanity check the fixed-point scaling before committing to the full pass.
_ts, _px, _sz = read_day(files[0])
print(files[0].name)
print(f'{len(_ts):,} ticks')
print('price ', _px[:4], f'  range {_px.min():,.2f} - {_px.max():,.2f}')
print('size  ', _sz[:4], f'  total {_sz.sum():,.1f} ETH')
del _ts, _px, _sz

In [ ]:
# Pass 1 -- daily ETH volume, needed for the dynamic (ADVV) thresholds.
totals = {}
for i, p in enumerate(files):
    _, _, sz = read_day(p)
    totals[day_of(p)] = float(sz.sum())
    if (i + 1) % 100 == 0:
        print(f'  {i + 1}/{len(files)} days')

dvol = pd.Series(totals).sort_index().asfreq('D')  # asfreq exposes genuine gaps as NaN
advv = {w: dvol.rolling(w, min_periods=w).mean().shift(1).ffill() for w in W_WINDOWS}

print(f'\n{len(dvol)} calendar days, {int(dvol.isna().sum())} with no data')
print(dvol.describe().to_string(float_format=lambda v: f'{v:,.0f}'))

fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(dvol.index, dvol.values, lw=0.5, color='0.75', label='daily ETH volume')
for w in W_WINDOWS:
    ax.plot(advv[w].index, advv[w].values, lw=1.3, label=f'ADVV W={w}')
ax.set_yscale('log')
ax.set_ylabel('ETH')
ax.set_title('Daily volume and trailing ADVV')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def acf1(x):
    d = x - x.mean()
    return float((d[:-1] * d[1:]).sum() / (d ** 2).sum())


def ljung_box_p(x, lags=10):
    d = x - x.mean()
    n, denom = len(d), (d ** 2).sum()
    q = sum(((d[:-k] * d[k:]).sum() / denom) ** 2 / (n - k) for k in range(1, lags + 1))
    return float(stats.chi2.sf(q * n * (n + 2), lags))


def tests(r, dur, days):
    """Normal marginal + independence, plus how long a bar takes to form.

    `kurt_se` is the approximate standard error of the kurtosis estimate, sqrt(24/n).
    Compare it to KURT_TOL before reading anything into a single period.
    """
    return {
        'bars': len(r),
        'bars_per_day': len(r) / days,
        'median_sec': float(np.median(dur)),
        'ex_kurtosis': float(stats.kurtosis(r)),
        'kurt_se': float(np.sqrt(24.0 / len(r))),
        'skew': float(stats.skew(r)),
        'jb_per_obs': float(stats.jarque_bera(r)[0] / len(r)),
        'acf1': acf1(r),
        'acf1_sq': acf1(r ** 2),
        'lb_sq_p': ljung_box_p(r ** 2),
    }


def subsample(x, n=PLOT_MAX, seed=0):
    if len(x) <= n:
        return x
    return x[np.sort(np.random.default_rng(seed).choice(len(x), n, replace=False))]

In [ ]:
# Pass 2 -- build bars for every spec. carry and prev live outside the period loop,
# so bars stay continuous across day and period boundaries alike.
carry = {s[0]: 0.0 for s in SPECS}
prev = None
rows, series = [], {}

for period, paths in groups.items():
    buf = {s[0]: {'ts': [], 'px': []} for s in SPECS}
    used = {s[0]: 0 for s in SPECS}

    for p in paths:
        ts, px, sz = read_day(p)
        if len(ts) == 0:
            continue
        d = day_of(p)

        for name, kind, a, b in SPECS:
            if kind == 'fixed':
                threshold = float(a)
            else:
                threshold = advv[b].get(d, np.nan) / a  # ADVV / target bars per day
            if not np.isfinite(threshold) or threshold <= 0:
                continue  # cold start: no trailing ADVV yet

            bts, bpx, carry[name] = close_bars(ts, px, sz, threshold, carry[name], prev)
            buf[name]['ts'].append(bts)
            buf[name]['px'].append(bpx)
            used[name] += 1

        prev = (ts[-1], px[-1])

    for name, *_ in SPECS:
        if not buf[name]['ts']:
            continue
        bts = np.concatenate(buf[name]['ts'])
        bpx = np.concatenate(buf[name]['px'])
        if len(bpx) < MIN_BARS:
            continue
        r = np.diff(np.log(bpx))
        dur = np.diff(bts) / 1e9
        rows.append({'period': period, 'spec': name, **tests(r, dur, used[name])})
        series[(period, name)] = (subsample(r).astype(np.float32),
                                  subsample(dur).astype(np.float32))

    del buf
    print(f'{period}: {len(paths)} days done')

res = pd.DataFrame(rows).sort_values(['period', 'spec']).reset_index(drop=True)

In [ ]:
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 1000)
print(res.to_string(index=False, float_format=lambda v: f'{v:,.4f}'))

In [ ]:
rank = (res.groupby('spec')[['ex_kurtosis', 'acf1_sq', 'bars_per_day', 'median_sec', 'kurt_se']]
        .mean().sort_values('ex_kurtosis'))
print(rank.to_string(float_format=lambda v: f'{v:,.4f}'))

# Kurtosis falls monotonically with threshold, so argmin always picks the largest
# bar in the grid. The useful choice is the FASTEST bar that is still normal enough.
ok = rank[rank['ex_kurtosis'].abs() <= KURT_TOL]
print(f'\nspecs within |excess kurtosis| <= {KURT_TOL}: {len(ok)} of {len(rank)}')
if len(ok):
    best = ok.sort_values('median_sec').iloc[0]
    print(f'fastest acceptable: {best.name}  '
          f'kurt {best.ex_kurtosis:.3f} (se {best.kurt_se:.3f})  '
          f'{best.bars_per_day:,.1f} bars/day  {best.median_sec / 60:,.1f} min/bar')
else:
    print('none -- raise KURT_TOL or extend the grid upward')

print('\nstability across periods (kurtosis std -- low means one setting holds):')
print(res.groupby('spec')['ex_kurtosis'].std()
      .sort_values().to_string(float_format=lambda v: f'{v:,.4f}'))

In [ ]:
# Fixed vs dynamic, through time.
#
# Plotted against integer positions, NOT the period strings. Matplotlib treats string
# x-values as categories ordered by first appearance, so a spec missing early periods
# (dropped for having < MIN_BARS) silently reorders the axis and draws lines straight
# across the figure.
periods_all = sorted(res['period'].unique())
pos = {p: i for i, p in enumerate(periods_all)}

fixed_present = sorted((s for s in res.spec.unique() if s.startswith('fix_')),
                       key=lambda s: int(s.split('_')[1]))
dyn_present = sorted(s for s in res.spec.unique() if s.startswith('dyn_'))
show = ([fixed_present[0], fixed_present[len(fixed_present) // 2], fixed_present[-1]]
        + dyn_present[:2])
print('showing:', show)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
for name in show:
    d = res[res.spec == name].sort_values('period')
    if d.empty:
        continue
    x = d['period'].map(pos)
    ax[0].plot(x, d['ex_kurtosis'], marker='o', ms=2.5, lw=1, label=name)
    ax[1].plot(x, d['acf1_sq'], marker='o', ms=2.5, lw=1, label=name)
    ax[2].plot(x, d['bars_per_day'], marker='o', ms=2.5, lw=1, label=name)

ax[0].axhline(0, color='k', ls='--', lw=1)
ax[0].axhline(KURT_TOL, color='g', ls=':', lw=1.2)
ax[0].set_yscale('symlog', linthresh=0.5)  # occasional spikes to 10+ otherwise flatten everything
ax[0].set_ylabel('excess kurtosis')
ax[0].set_title(f'1. Normal marginal (dotted: tol={KURT_TOL}; symlog above 0.5)')

ax[1].axhline(0, color='k', ls='--', lw=1)
ax[1].set_ylabel('acf1 of squared returns')
ax[1].set_title('2. Independence (0 = iid)')

ax[2].set_yscale('log')
ax[2].set_ylabel('bars/day')
ax[2].set_title('3. Bar rate -- fixed drifts, dynamic should not')

step = max(1, len(periods_all) // 14)
for a in ax:
    a.set_xticks(range(0, len(periods_all), step))
    a.set_xticklabels(periods_all[::step], rotation=90, fontsize=7)
    a.set_xlim(-0.5, len(periods_all) - 0.5)
    a.grid(alpha=0.3)
    a.legend(fontsize=6)
fig.tight_layout()
plt.show()

In [ ]:
def plot_period(period, names):
    avail = [n for n in names if (period, n) in series]
    if not avail:
        return
    fig, ax = plt.subplots(2, len(avail), figsize=(3.3 * len(avail), 6.4), squeeze=False)
    grid = np.linspace(-6, 6, 300)

    for j, name in enumerate(avail):
        r = series[(period, name)][0]
        z = (r - r.mean()) / r.std()
        ax[0, j].hist(z, bins=100, range=(-6, 6), density=True, color='#4C72B0')
        ax[0, j].plot(grid, stats.norm.pdf(grid), 'r', lw=1.2)
        ax[0, j].set_yscale('log')
        ax[0, j].set_ylim(1e-4, 1)
        ax[0, j].set_title(f'{name}\nkurt {float(stats.kurtosis(r)):.2f}  n={len(r):,}', fontsize=8)

        stats.probplot(z[:20000], dist='norm', plot=ax[1, j])
        ax[1, j].set_title('')
        ax[1, j].set_xlabel('theoretical', fontsize=8)
        ax[1, j].set_ylabel('sample', fontsize=8)

    fig.suptitle(f'{period}   log returns vs normal (top), QQ (bottom)')
    fig.tight_layout()
    plt.show()


def plot_durations(period, names):
    fig, ax = plt.subplots(figsize=(7.5, 4))
    for name in names:
        if (period, name) not in series:
            continue
        d = series[(period, name)][1]
        d = d[d > 0]
        ax.hist(np.log10(d), bins=80, histtype='step', lw=1.3, label=name)
    for sec, lab in [(60, '1min'), (600, '10min'), (3600, '1h')]:
        ax.axvline(np.log10(sec), color='k', ls=':', lw=0.8)
        ax.text(np.log10(sec), ax.get_ylim()[1] * 0.92, lab, fontsize=7)
    ax.set_xlabel('log10(bar duration / seconds)')
    ax.set_ylabel('count')
    ax.set_title(f'{period}   bar formation time')
    ax.legend(fontsize=7)
    plt.show()


for period in list(groups)[-PLOT_PERIODS:]:
    plot_period(period, show)
    plot_durations(period, show)

### How often should the threshold be retested live?

This cannot be answered in sample — re-selecting every period always looks best when you score on
the same data you selected from. It has to be a **walk-forward**: at each retest date choose the
spec using only the trailing `LOOKBACK` periods, apply it forward for `R` periods, and score what
it *actually* delivered over those forward periods. That is the number the live system would have
experienced.

**The selection rule matters more than the interval.** Selecting on lowest kurtosis is degenerate:
kurtosis decreases monotonically with threshold, so it always returns the largest spec in the grid,
never changes, and makes the retest question vacuous. Selection here is instead *the fastest spec
whose trailing kurtosis is within `KURT_TOL`* — that boundary genuinely moves as the market changes,
which is exactly what a retest is meant to catch.

Columns:

- **`breach_rate`** — fraction of forward periods where the chosen spec exceeded tolerance. This is
  the operational number: how much of the time your live bars were *not* normal enough.
- **`changes`** — how many times the threshold moved. Every change alters bar semantics, so features
  and models built on those bars shift scale underneath you. A policy marginally better on kurtosis
  but switching constantly is usually the worse choice.
- **`never`** — select once and never revisit. If retesting cannot beat this, the drift is not worth
  reacting to.

**In production, prefer a trigger to a calendar.** Monitor realized bars/day against target, or
rolling kurtosis, and retest when it breaches a band, with hysteresis so it does not oscillate.
That reacts to actual regime change instead of guessing a period. Treat the interval below as the
*maximum* time between checks, not as the mechanism itself.

Note the `dyn_*` specs already adapt daily, so for those the retest question reduces to `N` and `W`,
which are far more stable than an absolute ETH threshold. If the dynamic specs show a much lower
`breach_rate`, that is the argument for using them live.

In [ ]:
periods = sorted(res['period'].unique())
kurt = res.pivot_table(index='period', columns='spec', values='ex_kurtosis').reindex(periods)
speed = res.groupby('spec')['median_sec'].mean()  # seconds per bar; lower = faster


def select(window):
    """Fastest spec whose trailing mean |kurtosis| is within tolerance.

    Falls back to the least-bad spec when nothing qualifies. Selecting on argmin
    kurtosis instead would always return the largest threshold and never change.
    """
    m = window.abs().mean().dropna()
    ok = m[m <= KURT_TOL]
    return speed[ok.index].idxmin() if len(ok) else m.idxmin()


def walk_forward(retest, lookback=LOOKBACK):
    chosen, ks, breach, changes, secs = None, [], 0, 0, []
    for i, per in enumerate(periods):
        if i < lookback:
            continue
        if (i - lookback) % retest == 0:
            pick = select(kurt.iloc[i - lookback:i])
            changes += int(chosen is not None and pick != chosen)
            chosen = pick
        v = kurt.loc[per, chosen]
        if np.isfinite(v):
            ks.append(abs(v))
            breach += int(abs(v) > KURT_TOL)
            secs.append(speed[chosen])
    if not ks:
        return None
    return {'oos_abs_kurtosis': float(np.mean(ks)), 'breach_rate': breach / len(ks),
            'changes': changes, 'min_per_bar': float(np.mean(secs)) / 60,
            'periods_scored': len(ks)}


out = []
for retest in (1, 2, 3, 6, 12, 24):
    if retest > len(periods) - LOOKBACK:
        continue
    r = walk_forward(retest)
    if r:
        out.append({'retest_periods': str(retest), **r})

r = walk_forward(len(periods))
if r:
    out.append({'retest_periods': 'never', **r})
out.append({'retest_periods': 'oracle', 'oos_abs_kurtosis': float(kurt.abs().min(axis=1).mean()),
            'breach_rate': np.nan, 'changes': np.nan, 'min_per_bar': np.nan,
            'periods_scored': len(periods)})

wf = pd.DataFrame(out)
print(f'PERIOD={PERIOD}  lookback={LOOKBACK}  tolerance={KURT_TOL}\n')
print(wf.to_string(index=False, float_format=lambda v: f'{v:,.4f}'))

real = wf[wf['retest_periods'] != 'oracle'].dropna(subset=['breach_rate'])
if len(real):
    b = real.loc[real['breach_rate'].idxmin()]
    print(f"\nlowest breach rate: retest every {b['retest_periods']} period(s) "
          f"-- {b['breach_rate']:.1%} of periods out of tolerance, "
          f"{int(b['changes'])} threshold changes")